# 01 · Data audit

**Phase NS1 · Scope & data audit**

Goal: for every dataset, confirm what columns it has, what geography it uses, what time period it covers and what its limits are. **No cleaning or modelling yet.** Record findings in `data/README.md` and decisions in `docs/decision_log.md`.

Before running: download the files listed in `data/README.md` into `data/raw/`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import geopandas as gpd

from src.config import DATA_RAW, CRS_SVY21

pd.set_option('display.max_columns', 50)
sorted(p.name for p in DATA_RAW.iterdir())

## A reusable audit helper

This prints the same checks for every table so the audits are comparable.

In [ ]:
def audit(df, name):
    print(f'=== {name} ===')
    print(f'rows: {len(df):,} | columns: {df.shape[1]}')
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_%': (df.isna().mean() * 100).round(1),
        'unique': df.nunique(),
        'example': df.iloc[0] if len(df) else None,
    })
    print(f'duplicate rows: {df.duplicated().sum():,}')
    return summary

## 1. URA Master Plan 2019 subzone boundaries

Questions to answer:
- How many subzones and planning areas are there?
- Which column holds the subzone name? The planning area name?
- What CRS is the file in?

In [ ]:
# Change the file name to match your download
subzones = gpd.read_file(DATA_RAW / 'MasterPlan2019SubzoneBoundaryNoSeaGEOJSON.geojson')
print('CRS:', subzones.crs)
audit(subzones.drop(columns='geometry'), 'URA subzones')

In [ ]:
subzones.plot(figsize=(10, 6), edgecolor='white', linewidth=0.3)

**Notes:**
- Subzone name column: 
- Planning area column: 
- CRS: 
- Count of subzones: 

## 2. Census 2020 population by subzone

Questions to answer:
- Is the table one row per subzone, or one row per subzone and age group?
- How are subzone names written? Do they match the URA file?

In [ ]:
# Change the file name to match your download
pop = pd.read_csv(DATA_RAW / 'census2020_population_subzone.csv')
audit(pop, 'Census 2020 population')

In [ ]:
pop.head(10)

**Notes:**
- Table shape (wide or long): 
- Name format compared with URA: 
- Rows that look like totals and need removing: 

## 3. LTA MRT/LRT station exits

Questions to answer:
- What coordinate system are the exits in?
- How many exits and stations are there?

In [ ]:
# Change the file name to match your download
exits = gpd.read_file(DATA_RAW / 'TrainStationExit.geojson')
print('CRS:', exits.crs)
audit(exits.drop(columns='geometry'), 'LTA station exits')

## 4. ACRA business entities

Questions to answer:
- Which columns hold the SSIC code, registration date, status and postal code?
- How many entities have an F&B SSIC code (starting 561 or 563)?
- What date range do registrations cover?

Postal codes must be read as text so leading zeros are kept.

In [ ]:
# Change the file name to match your download
acra = pd.read_csv(DATA_RAW / 'acra_entities.csv', dtype=str, low_memory=False)
audit(acra, 'ACRA entities')

In [ ]:
from src.data_processing import is_fnb

# Replace with the real SSIC column name once you've found it
ssic_col = 'primary_ssic_code'
fnb = acra[acra[ssic_col].apply(is_fnb)]
print(f'F&B entities: {len(fnb):,} of {len(acra):,}')

## 5. Summary of findings

Copy these into `data/README.md`.

| Dataset | Key columns | Geography | Time span | Main caveat |
|---|---|---|---|---|
| URA subzones | | | | |
| Census 2020 | | | | |
| LTA exits | | | | |
| ACRA | | | | |

## 6. Decisions made today

Add each to `docs/decision_log.md`.